# Materials landscape analysis

We’ll analyse how alloy composition shapes an experimentally measured property, from preparing composition data to exploring local optima, ruggedness and interactions. A tungsten–rhenium–osmium alloy library will serve as our example.

## 1. Importing necessary dependencies

Install GraphFLA with pip, then import the dependencies below.

In [ ]:
!pip install graphfla

In [1]:
import graphfla
from graphfla.landscape import OrdinalLandscape
import pandas as pd

## 2. Loading the experimental dataset

For our example, we use the W–Re–Os alloy library from [Wang et al. (2026)](https://doi.org/10.1038/s41467-025-67301-7). Combinatorial additive manufacturing produced a library of 496 compositions, sampled in steps of 3 atomic percent.

We’ll maximize **Vickers hardness at 1000 °C**, using the measurements curated in the repository. The three elemental percentages sum to 100.

| Column | Meaning |
|---|---|
| `composition_id` | Label identifying the alloy composition. |
| `W` | Tungsten content, in atomic percent. |
| `Re` | Rhenium content, in atomic percent. |
| `Os` | Osmium content, in atomic percent. |
| `H1000_HV` | Vickers hardness at 1000 °C, in HV. |

Let’s load these columns and inspect the first few alloys.

In [2]:
data_url = (
    "https://raw.githubusercontent.com/COLA-Laboratory/GraphFLA/"
    "main/data/Materials/WReOs/simplex.csv"
)
df = pd.read_csv(data_url)[["composition_id", "W", "Re", "Os", "H1000_HV"]]
df.head()

,composition_id,W,Re,Os,H1000_HV
0,W3Re3Os94,3,3,94,411
1,W3Re6Os91,3,6,91,423
2,W3Re9Os88,3,9,88,442
3,W3Re12Os85,3,12,85,460
4,W3Re15Os82,3,15,82,454


## 3. Preparing the inputs

To construct a landscape, GraphFLA only needs two inputs—much like the features `X` and targets `y` used to train a machine-learning model:

- `X`: the experimental configurations, as a DataFrame or a two-dimensional NumPy array.
- `f`: one measured outcome per configuration, as a Series, list or one-dimensional array.

The rows of `X` and `f` must refer to the same configurations in the same order.


Only two composition coordinates are independent: `Os = 100 - W - Re`. We therefore use `W` and `Re` as `X`, and high-temperature hardness as `f`.

Including all three percentages would prevent a one-coordinate change from preserving the composition total. The alloy label is an identifier, so it also stays out of `X`.

In [3]:
X = df[['W', 'Re']]
f = df['H1000_HV']
X.head()

,W,Re
0,3,3
1,3,6
2,3,9
3,3,12
4,3,15


## 4. Constructing the landscape

We can use `OrdinalLandscape` because the composition levels are ordered. Neighbours differ by one sampled level in W or Re, with the other coordinate fixed. Os adjusts to preserve the total of 100 atomic percent.

Under this chosen rule, a step exchanges 3 atomic percent between Os and either W or Re. A direct W–Re exchange requires two steps; the resulting landscape statistics refer to this coordinate grid.

We set `maximize=True` because higher hardness is our objective.

In [4]:
landscape = OrdinalLandscape(maximize=True)
landscape.build_from_data(X, f)

Building Landscape from data...


Preparing Ordinal input...


Ordinal input preparation complete. Detected 2 variable(s).


 - Handling missing values and duplicates...


 - Checking for missing values...


...

   - Found 36 local optima (36 member nodes: 0 plateau-LOs + 36 single-point LOs).


 - Determining global optimum...


   - Global optimum found at index 436 with fitness 1265.0000.


Landscape built successfully.



OrdinalLandscape(maximize=True)

## 5. Inspecting the landscape

Let’s first look at what we’ve built. Printing the landscape shows its variables, configurations, improving edges and local optima:

In [5]:
print(landscape)

OrdinalLandscape(kind='ordinal'): 2 variables, 496 configurations, 911 edges, 36 local optima


For a closer look at individual alloy compositions, we can call `get_data()`. The selected outcome appears as `fitness`; `out_degree` counts directly improving moves, `in_degree` counts incoming moves from lower-scoring neighbours, and `is_lo` indicates local-optimum membership.

In [6]:
landscape.get_data()

,W,Re,fitness,plateau_id,plateau_size,in_degree,out_degree,is_lo
0,3,3,411.0,-1,1,0,2,False
1,3,6,423.0,-1,1,1,2,False
2,3,9,442.0,-1,1,1,2,False
3,3,12,460.0,-1,1,2,1,False
4,3,15,454.0,-1,1,1,2,False
...,...,...,...,...,...,...,...,...
491,87,6,504.0,-1,1,3,1,False
492,87,9,502.0,-1,1,0,2,False
493,90,3,431.0,15,2,1,1,False
494,90,6,431.0,15,2,0,1,False


We can also summarise the hardness measurements with pandas:

In [7]:
landscape.get_data()["fitness"].describe()

count     496.000000
mean      609.294355
std       296.659404
min       201.000000
25%       396.000000
50%       453.500000
75%       826.750000
max      1265.000000
Name: fitness, dtype: float64

To inspect the alloy with the highest high-temperature hardness, we can use the global-optimum node index:

In [8]:
landscape[landscape.go_index]

{'W': np.int64(63),
 'Re': np.int64(21),
 'fitness': np.float64(1265.0),
 'plateau_id': -1,
 'plateau_size': 1,
 'in_degree': 4,
 'out_degree': 0,
 'is_lo': True}

The node reports W and Re; the remaining percentage is `Os = 100 - W - Re`. This selects for hardness alone. The paper’s broader materials selection also considered hardness retention and plasticity.

## 6. Analysing the landscape

Next, we’ll count local optima, measure ruggedness, examine interactions and assess the trend toward the best observed configuration.

In [9]:
from graphfla import analysis

### 6.1 Number of local optima

We can start with `landscape.n_lo`. A local optimum has no improving route out of its immediate neighbourhood, including neutral moves within a plateau. Each connected neutral optimum plateau counts once. Multiple local optima mean that successive improvements can lead to different endpoints.

In [10]:
print(f"Number of local optima: {landscape.n_lo}")

Number of local optima: 36


### 6.2 Fitness autocorrelation

To explore ruggedness, we can measure how similar the scores remain along random walks through neighbouring configurations. Positive autocorrelation indicates persistence of similar scores across nearby steps; lower values indicate less persistence under the same walk settings.

We use 200 walks of 20 visited configurations and a fixed seed. The walks follow stored edges in either direction and exclude separately stored neutral pairs.

In [11]:
walk_autocorrelation = analysis.autocorrelation(
    landscape,
    walk_length=20,
    walk_times=200,
    lag=1,
    seed=42,
)
print(f"Lag-1 fitness autocorrelation: {walk_autocorrelation:.4f}")

Lag-1 fitness autocorrelation: 0.9366


### 6.3 Interactions between composition changes

Increasing W can have a different effect on hardness at different Re levels. We can examine this context dependence with `classify_epistasis()`. In this setting, *epistasis* refers to interactions on the chosen composition grid, with Os adjusting as the balance element.

We can explore these patterns with `classify_epistasis()`, which classifies four-configuration graph motifs:

| Category | Interpretation |
|---|---|
| `magnitude` | Changes retain their direction of effect across contexts. |
| `sign` | One change reverses its direction of effect across contexts. |
| `reciprocal_sign` | Both changes reverse their direction of effect across contexts. |

The output gives their proportions among the detected motifs. The `positive` and `negative` fields form a separate partition.

In [12]:
interactions = analysis.classify_epistasis(landscape, seed=42)
interactions

EpistasisClassification(magnitude=0.6228287841191067, sign=0.2704714640198511, reciprocal_sign=0.10669975186104218, positive=0.7121588089330024, negative=0.28784119106699757)

### 6.4 Fitness-distance correlation

Finally, we can use `fdc` to check whether higher-scoring configurations tend to lie closer to the best observed configuration—a simple view of navigability.


Distance is the total number of sampled composition steps along W and Re. It uses the same two-coordinate geometry as the neighbourhood rule.

We use Spearman correlation to compare ranks. A negative value means higher scores tend to occur nearer the optimum; a value near zero indicates little monotonic association. This trend does not guarantee an improving path from every configuration.

In [13]:
fitness_distance_r = analysis.fdc(landscape, method="spearman")
print(f"Fitness-distance correlation: {fitness_distance_r:.4f}")

 - Calculating distances to global optimum using mixed_distance...


   - Distances to GO calculated and added as node attribute 'dist_go'.


Fitness-distance correlation: -0.8687


## 7. Running several analyses together

Once we’ve tried the metrics individually, we can collect them with `analysis.profile()`. Here, we combine the ruggedness measure and fitness-distance correlation in a pandas Series. Setting `on_error="raise"` displays errors immediately.

In [14]:
analysis.profile(
    landscape,
    include=["autocorrelation", "fdc"],
    params={"autocorrelation": {"walk_times": 200}},
    seed=42,
    on_error="raise",
    progress=False,
)

autocorrelation    0.936634
fdc               -0.868738
dtype: float64

## 8. Analysis reference

For further exploration, we can choose from the **30 analysis functions** below. Each takes a built `landscape` as its first argument. In these functions, *fitness* refers to the outcome selected as `f`.

Walsh–Hadamard and higher-order regression are included for reference. Their cost grows rapidly with the number of variables and levels, so we leave them out of the worked examples.

| Group | Function | Result |
| --- | --- | --- |
| Fitness | `fitness_distribution` | Dictionary of distribution summaries, including skewness, kurtosis, CV and Cauchy location. |
| Fitness | `fitness_effect_distribution` | List of signed effects for one mutation across matched backgrounds. |
| Ruggedness | `local_optima_ratio` | Local-optimum count divided by configuration count. |
| Ruggedness | `r_s_ratio` | Additive-model residual roughness divided by mean absolute coefficient. |
| Ruggedness | `autocorrelation` | Fitness autocorrelation along sampled walks. |
| Ruggedness | `gradient_intensity` | Mean absolute change on improving edges divided by mean fitness. |
| Correlation | `fdc` | Fitness-distance correlation: Spearman or Pearson. |
| Correlation | `neighbor_fitness_correlation` | Correlation between fitness and mean neighbour fitness: Pearson, Spearman or Kendall. |
| Correlation | `basin_fitness_correlation` | Correlation between peak fitness and greedy basin size: Spearman or Pearson. |
| Correlation | `fitness_flattening_index` | Trend in successive fitness gains along greedy walks: Spearman or Pearson. |
| Navigability | `global_optima_accessibility` | Fraction of configurations with an improving route to the global optimum. |
| Navigability | `local_optima_accessibility` | DataFrame of accessibility values for specified local optima. |
| Navigability | `mean_distance_to_global_optimum` | Mean sequence/configuration distance to the global optimum. |
| Navigability | `mean_distance_to_local_optima` | DataFrame of mean sequence/configuration distances to specified local optima. |
| Navigability | `mean_path_length_to_global_optimum` | Mean shortest graph-path length, excluding unreachable starts. |
| Navigability | `mean_path_length_to_local_optima` | DataFrame of shortest-path summaries, excluding unreachable starts. `lo=None` selects the global optimum. |
| Robustness | `neutrality` | Fraction of neighbour pairs within the fitness-difference threshold. |
| Robustness | `evolvability_enhancing_fraction` | Fraction of all directed neighbour pairs classified as EE using non-focal neighbourhoods and BH-corrected tests. |
| Robustness | `single_mutation_effects` | DataFrame of mutation-pair effects and one-sided tests at one position; positive or negative tests. |
| Robustness | `all_mutation_effects` | DataFrame of mutation-pair effects and one-sided tests at all positions. |
| Epistasis | `idiosyncratic_index` | Normalized spread of one mutation’s effects across backgrounds. |
| Epistasis | `global_idiosyncratic_index` | Mean idiosyncratic index across eligible mutations. |
| Epistasis | `gamma` | Correlation of mutation effects across matched two-site backgrounds. |
| Epistasis | `gamma_star` | Consistency of mutation-effect signs across backgrounds. |
| Epistasis | `diminishing_returns_index` | Association of background fitness with mean beneficial gain: Pearson, Spearman or regression slope. |
| Epistasis | `increasing_costs_index` | Association of background fitness with mean deleterious cost: Pearson, Spearman or regression slope. |
| Epistasis | `higher_order_epistasis` | Per-order cumulative R² and incremental fit gains; accepts an existing W-H result without refitting. |
| Epistasis | `walsh_hadamard` | Coefficients and an order summary with fit gains and the fitted-model variance spectrum. |
| Epistasis | `classify_epistasis` | Object with magnitude, sign, reciprocal-sign, positive and negative epistasis proportions. |
| Epistasis | `extradimensional_bypass` | Object with bypass proportion, mean length and motif counts. |

### Data source

Wang W., Liu J., Li C. et al. (2026). [*High-throughput discovery of ultrahigh-temperature multi-principal element alloys by combinatorial additive manufacturing*. Nature Communications 17, 668](https://doi.org/10.1038/s41467-025-67301-7).

The repository CSV contains composition and property measurements curated from the paper’s Source Data workbook.